# 01 — Data Extraction

Parse all Freddie Mac SF LLD quarterly zip files (2018–2025) into parquet files,
then build the model-specific subsets defined in spec §4.6.

**Run All** executes the full pipeline end-to-end.  
Already-processed files are skipped automatically.

Expected final outputs in `data/processed/`:
- `origination_YYYYQN.parquet` × 31 files
- `performance_YYYYQN.parquet` × 31 files
- `origination_all.parquet`
- `panel_logistic_2021_2025.parquet`
- `panel_cph_2018_2025.parquet`

> **Note — 16 GB RAM machines:** `performance_all.parquet` (requires ~20 GB) and
> `merged_panel.parquet` (requires ~15 GB) are skipped. The pipeline builds the
> model subsets directly from quarterly files (peak ~4 GB).


In [1]:
import logging
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, str(Path('..') / 'src'))
import data_extraction as de

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

PROCESSED = Path('..') / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

## Phase 2a — Parse all origination files

In [2]:
orig_all_path = de.parse_all_origination(years=range(2018, 2026), out_dir=PROCESSED)
print(f'origination_all.parquet → {orig_all_path}')

INFO Already exists, skipping: ../data/processed/origination_2018Q1.parquet
INFO Already exists, skipping: ../data/processed/origination_2018Q2.parquet
INFO Already exists, skipping: ../data/processed/origination_2018Q3.parquet
INFO Already exists, skipping: ../data/processed/origination_2018Q4.parquet
INFO Already exists, skipping: ../data/processed/origination_2019Q1.parquet
INFO Already exists, skipping: ../data/processed/origination_2019Q2.parquet
INFO Already exists, skipping: ../data/processed/origination_2019Q3.parquet
INFO Already exists, skipping: ../data/processed/origination_2019Q4.parquet
INFO Already exists, skipping: ../data/processed/origination_2020Q1.parquet
INFO Already exists, skipping: ../data/processed/origination_2020Q2.parquet
INFO Already exists, skipping: ../data/processed/origination_2020Q3.parquet
INFO Already exists, skipping: ../data/processed/origination_2020Q4.parquet
INFO Already exists, skipping: ../data/processed/origination_2021Q1.parquet
INFO Already

origination_all.parquet → ../data/processed/origination_all.parquet


## Phase 2b — Parse all performance files

In [3]:
# Parse quarterly performance files — skips any already on disk.
# performance_all.parquet (concat of all 31 files) requires ~20 GB RAM and is
# NOT needed: create_model_subsets() builds panels directly from quarterly files.
from itertools import product

for year, qtr in product(range(2018, 2026), range(1, 5)):
    zip_path = Path('..') / f'historical_data_{year}' / f'historical_data_{year}Q{qtr}.zip'
    if zip_path.exists():
        de.parse_performance_quarter(year, qtr, zip_path, PROCESSED)

quarterly_perf = sorted(PROCESSED.glob('performance_????Q?.parquet'))
print(f'Quarterly performance files ready: {len(quarterly_perf)}')


INFO Already exists, skipping: ../data/processed/performance_2018Q1.parquet
INFO Already exists, skipping: ../data/processed/performance_2018Q2.parquet
INFO Already exists, skipping: ../data/processed/performance_2018Q3.parquet
INFO Already exists, skipping: ../data/processed/performance_2018Q4.parquet
INFO Already exists, skipping: ../data/processed/performance_2019Q1.parquet
INFO Already exists, skipping: ../data/processed/performance_2019Q2.parquet
INFO Already exists, skipping: ../data/processed/performance_2019Q3.parquet
INFO Already exists, skipping: ../data/processed/performance_2019Q4.parquet
INFO Already exists, skipping: ../data/processed/performance_2020Q1.parquet
INFO Already exists, skipping: ../data/processed/performance_2020Q2.parquet
INFO Already exists, skipping: ../data/processed/performance_2020Q3.parquet
INFO Already exists, skipping: ../data/processed/performance_2020Q4.parquet
INFO Already exists, skipping: ../data/processed/performance_2021Q1.parquet
INFO Already

Quarterly performance files ready: 31


## Phase 2c — Merge panel and create model-specific subsets

In [4]:
# build_merged_panel() is skipped — it produces a ~10 GB file not needed for modeling.
# create_model_subsets() detects merged_panel.parquet is absent and builds both
# model panels directly from quarterly files (peak ~4 GB RAM).
subsets = de.create_model_subsets(out_dir=PROCESSED)
print(f'panel_logistic → {subsets["logistic"]}')
print(f'panel_cph      → {subsets["cph"]}')


INFO Both model subsets already exist, skipping.


panel_logistic → ../data/processed/panel_logistic_2021_2025.parquet
panel_cph      → ../data/processed/panel_cph_2018_2025.parquet


## Verification — Row counts, schema, default rates

In [5]:
# Shape check via parquet metadata — no data loaded into RAM.
import pyarrow.parquet as pq

for label, path in [('panel_logistic', subsets['logistic']), ('panel_cph', subsets['cph'])]:
    meta = pq.read_metadata(path)
    print(f'{label}: {meta.num_rows:,} rows × {meta.num_columns} cols')
print('(expect 63 cols = 32 perf + 32 orig - 1 shared loan_id)')


panel_logistic: 1,223 rows × 64 cols
panel_cph: 60,585,348 rows × 64 cols
(expect 63 cols = 32 perf + 32 orig - 1 shared loan_id)


In [6]:
# Loan count per origination year — uses origination_all.parquet (281 MB, fits in RAM).
# This is a loan count, not a performance-row count, but confirms cohort coverage.
orig = pd.read_parquet(PROCESSED / 'origination_all.parquet')
orig['orig_year'] = pd.to_datetime(
    orig['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
).dt.year
print('Loans originated per year (spec §10 Phase 2 check):')
print(orig.groupby('orig_year').size().rename('loan_count').to_string())
del orig  # free RAM before next cell


Loans originated per year (spec §10 Phase 2 check):
orig_year
2018    1103972
2019    1613891
2020    3419282
2021    4391081
2022    1988812
2023     942481
2024    1008180
2025     812875


In [ ]:
# Loan-level default rate on 2021-2025 panel — streamed row-group by row-group.
# The panel is ~329M rows; loading loan_id whole would need ~13 GB, so we stream.
import pyarrow.parquet as pq

seen, defaulted = set(), set()
pf = pq.ParquetFile(subsets['logistic'])
for b in pf.iter_batches(batch_size=4_000_000,
                         columns=['loan_id', 'zero_balance_code', 'delinquency_status']):
    df = b.to_pandas()
    seen.update(df['loan_id'].values)
    dq = pd.to_numeric(df['delinquency_status'], errors='coerce').fillna(0)
    is_def = df['zero_balance_code'].isin(['03', '06', '09']) | (dq >= 3)
    defaulted.update(df.loc[is_def, 'loan_id'].values)

rate = len(defaulted) / len(seen) if seen else 0
print(f'Unique loans: {len(seen):,}')
print(f'Defaulted loans (terminal 03/06/09 or ever 90+ DPD): {len(defaulted):,}')
print(f'Loan-level default rate: {rate:.4%}')
print('Note: dominated by the ever-90+-DPD condition; terminal-only default is ~0.01%')
print('on these recent 2021-2025 vintages. Choose the default definition in notebook 03.')


In [ ]:
# Terminal foreclosure/REO events (zero_balance_code='09') by orig_year — streamed.
from collections import Counter

fc = Counter()
pf = pq.ParquetFile(subsets['cph'])
for b in pf.iter_batches(batch_size=4_000_000, columns=['zero_balance_code', 'first_payment_date']):
    df = b.to_pandas()
    hit = df[df['zero_balance_code'] == '09']
    yr = pd.to_datetime(hit['first_payment_date'].astype(str), format='%Y%m',
                        errors='coerce').dt.year
    fc.update(yr.dropna().astype(int).value_counts().to_dict())

print('REO/foreclosure rows (zero_balance_code=09) by origination year:')
for y in sorted(fc):
    print(f'  {y}: {fc[y]:,}')
print('\nIf 2018-2020 shows zero → zero_balance_code parsing error (spec §10 Phase 2 check)')


In [ ]:
# Field completeness on key modeling columns — streamed weighted non-null fraction.
key_cols = ['credit_score', 'ltv', 'dti', 'orig_interest_rate',
            'current_upb', 'delinquency_status', 'loan_age']
non_null = {c: 0 for c in key_cols}
total = 0
pf = pq.ParquetFile(subsets['cph'])
for b in pf.iter_batches(batch_size=4_000_000, columns=key_cols):
    df = b.to_pandas()
    total += len(df)
    for c in key_cols:
        non_null[c] += df[c].notna().sum()

print('Field completeness (non-null %):')
for c in key_cols:
    print(f'  {c:<22} {non_null[c]/total:.1%}')


In [10]:
# Output file inventory
print('Files in data/processed/:')
for parquet_file in sorted(PROCESSED.glob('*.parquet')):
    size_mb = parquet_file.stat().st_size / 1e6
    print(f'  {parquet_file.name:<45} {size_mb:>7.1f} MB')

Files in data/processed/:
  origination_2018Q1.parquet                        5.5 MB
  origination_2018Q2.parquet                        6.6 MB
  origination_2018Q3.parquet                        6.1 MB
  origination_2018Q4.parquet                        5.3 MB
  origination_2019Q1.parquet                        5.2 MB
  origination_2019Q2.parquet                        7.4 MB
  origination_2019Q3.parquet                        9.7 MB
  origination_2019Q4.parquet                        9.8 MB
  origination_2020Q1.parquet                        9.3 MB
  origination_2020Q2.parquet                       16.6 MB
  origination_2020Q3.parquet                       20.9 MB
  origination_2020Q4.parquet                       22.6 MB
  origination_2021Q1.parquet                       21.5 MB
  origination_2021Q2.parquet                       17.0 MB
  origination_2021Q3.parquet                       18.5 MB
  origination_2021Q4.parquet                       15.5 MB
  origination_2022Q1.parquet  